In [1]:
!pip install fastapi uvicorn python-multipart pydantic -q

In [2]:
!pip install nest_asyncio

In [3]:
from fastapi import FastAPI
from pydantic import BaseModel, Field
from typing import Optional, List
import pandas as pd
import joblib
import logging
import uvicorn
from google.colab import drive
import threading
import requests
import time
from google.colab.output import eval_js
import json
import os

# конфиг
PORT = 8000
BASE_URL = f"http://localhost:{PORT}"
DRIVE_FOLDER = '/content/drive/MyDrive/heart_model'
DEFAULT_CSV_FILENAME = 'heart_test.csv'

drive.mount('/content/drive', force_remount=False)
!cp '{DRIVE_FOLDER}/heart_preprocessor.py' .
!cp '{DRIVE_FOLDER}/heart_predictor.py' .

from heart_preprocessor import DataPreprocessor
from heart_predictor import HeartRiskPredictor

preprocessor_path = f'{DRIVE_FOLDER}/preprocessor.pkl'
model_path = f'{DRIVE_FOLDER}/model.pkl'

try:
    # Загружаем preprocessing pipeline
    preprocessor = joblib.load(preprocessor_path)
    print("Preprocessor загружен успешно")

    # Загружаем обученную модель
    model = joblib.load(model_path)
    print("Модель загружена успешно")

except FileNotFoundError:
    print("Файлы с моделью не найдены")
    print(f"Preprocessor: {preprocessor_path}")
    print(f"Model: {model_path}")
except Exception as e:
    print(f"Ошибка загрузки файлов модели: {e}")

predictor = HeartRiskPredictor(
    preprocessor_path=preprocessor_path,
    model_path=model_path
)


# Создание приложения
app = FastAPI(title="Heart Risk Prediction API")

@app.get("/")
async def root():
    return {
        "message": "Heart Risk Prediction API",
        "status": "active",
        "version": "1.0.0",
        "base_url": BASE_URL,
        "port": PORT
    }

@app.get("/health")
async def health_check():
    if predictor is None:
        return {"status": "error", "message": "Model not loaded"}
    return {"status": "healthy", "message": "API and model are ready"}

class PatientData(BaseModel):
    age: float
    gender: str
    diabetes: float
    family_history: float
    smoking: float
    obesity: float
    alcohol_consumption: float
    previous_heart_problems: float
    medication_use: float
    bmi: float
    systolic_blood_pressure: float
    diastolic_blood_pressure: float
    income: float
    cholesterol: float
    heart_rate: float
    exercise_hours_per_week: float
    stress_level: float
    sedentary_hours_per_day: float
    triglycerides: float
    physical_activity_days_per_week: float
    sleep_hours_per_day: float
    diet: float

    class Config:
        populate_by_name = True

@app.post("/predict")
async def predict_risk(patient_data: PatientData):
    """Предсказание риска сердечных заболеваний по данным в ячейке"""
    try:
        if predictor is None:
            return {"status": "error", "message": "Model not loaded"}

        # Преобразуем данные в словарь
        input_dict = patient_data.model_dump()

        # Создаем DataFrame
        input_df = pd.DataFrame([input_dict])

        # Получаем предсказание
        prediction_result = predictor.predict(input_df)

        # Обрабатываем результат
        if hasattr(prediction_result, 'tolist'):
            prediction_result = prediction_result.tolist()

        return {
            "status": "success",
            "prediction": prediction_result,
            "input_data": input_dict,
            "timestamp": time.time()
        }

    except Exception as e:
        return {
            "status": "error",
            "message": f"Prediction failed: {str(e)}",
            "timestamp": time.time()
        }



@app.post("/predict-csv")
async def predict_from_drive_csv(filename: str = DEFAULT_CSV_FILENAME):
    """Предсказание риска сердечных заболеваний из CSV файла в Google Drive"""
    try:

        if predictor is None:
            return {"status": "error", "message": "Model not loaded"}

        file_path = f'{DRIVE_FOLDER}/{filename}'
        df = pd.read_csv(file_path)
        print(f"Файл загружен. Строк: {len(df)}, колонок: {len(df.columns)}")

        # Получаем предсказания
        prediction_result = predictor.predict(df)

        # Извлекаем только предсказанные классы
        if isinstance(prediction_result, dict) and 'predictions' in prediction_result:
            predictions = prediction_result['predictions']

            # Преобразуем в список если это numpy array
            if hasattr(predictions, 'tolist'):
                predictions_list = predictions.tolist()
            else:
                predictions_list = list(predictions)



        # Создаем результат в формате JSON
        results = []
        for i in range(len(df)):
            patient_result = {
                "patient_id": i,
                "prediction": predictions_list[i] if i < len(predictions_list) else None
            }
            results.append(patient_result)

        print(f"Успешно создано {len(results)} результатов")

        return {
            "status": "success",
            "filename": filename,
            "total_patients": len(df),
            "predictions": results,
            "timestamp": time.time()
        }

    except Exception as e:
        print(f"Ошибка в эндпоинте: {str(e)}")

        return {
            "status": "error",
            "message": f"Error: {str(e)}"
        }


# Запускаем сервер в фоне
def run_server():
    uvicorn.run(app, host="0.0.0.0", port=PORT)

thread = threading.Thread(target=run_server, daemon=True)
thread.start()

# Ждем и получаем URL
time.sleep(5)
try:
    public_url = eval_js("google.colab.kernel.proxyPort(8000)")
    print(f"Сервер запущен: {public_url}")
    print(f"Health check: {public_url}/health")
except Exception as e:
    print(f"URL ошибка: {e}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Preprocessor загружен успешно
Модель загружена успешно


INFO:     Started server process [2499]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:8000 (Press CTRL+C to quit)


Сервер запущен: https://8000-m-s-2q38lr41s3k0t-c.us-west1-0.prod.colab.dev
Health check: https://8000-m-s-2q38lr41s3k0t-c.us-west1-0.prod.colab.dev/health


In [4]:
# Тест на 1 пациенте
test_patient = {
    "age": 0.4943820224719101,
    "gender": "Male",
    "diabetes": 0.0,
    "family_history": 1.0,
    "smoking": 1.0,
    "obesity": 1.0,
    "alcohol_consumption": 1.0,
    "previous_heart_problems": 1.0,
    "medication_use": 0.0,
    "bmi": 0.4591759037537125,
    "systolic_blood_pressure": 0.2129032258064515,
    "diastolic_blood_pressure": 0.7093023255813953,
    "income": 0.105948008517571,
    "cholesterol": 0.732142857142857,
    "heart_rate": 0.074243813015582,
    "exercise_hours_per_week": 0.5355049297181428,
    "stress_level": 8.0,
    "sedentary_hours_per_day": 0.2257038936087668,
    "triglycerides": 0.9792207792207792,
    "physical_activity_days_per_week": 3.0,
    "sleep_hours_per_day": 0.3333333333333333,
    "diet": 1.0
}

print("Тестирование метода predict...")

try:
    response = requests.post(f"{BASE_URL}/predict", json=test_patient)
    print(f"Статус код: {response.status_code}")

    if response.status_code == 200:
        result = response.json()
        print("Success")
        print(json.dumps(result, indent=2, ensure_ascii=False))
    else:
        print(f"Ошибка: {response.text}")

except Exception as e:
    print(f"Исключение: {e}")

Тестирование метода predict...
INFO:     127.0.0.1:52432 - "POST /predict HTTP/1.1" 200 OK
Статус код: 200
Success
{
  "status": "success",
  "prediction": {
    "predictions": [
      1
    ],
    "probabilities": [
      0.355
    ],
    "risk_level": [
      "high"
    ],
    "threshold": 0.34,
    "model_type": "RandomForestClassifier"
  },
  "input_data": {
    "age": 0.4943820224719101,
    "gender": "Male",
    "diabetes": 0.0,
    "family_history": 1.0,
    "smoking": 1.0,
    "obesity": 1.0,
    "alcohol_consumption": 1.0,
    "previous_heart_problems": 1.0,
    "medication_use": 0.0,
    "bmi": 0.4591759037537125,
    "systolic_blood_pressure": 0.2129032258064515,
    "diastolic_blood_pressure": 0.7093023255813953,
    "income": 0.105948008517571,
    "cholesterol": 0.732142857142857,
    "heart_rate": 0.074243813015582,
    "exercise_hours_per_week": 0.5355049297181428,
    "stress_level": 8.0,
    "sedentary_hours_per_day": 0.2257038936087668,
    "triglycerides": 0.9792207

In [5]:
# Тест на файле со списком пациентов в csv
response = requests.post(
        f"{BASE_URL}/predict-csv",
        params={"filename": "heart_test.csv"},
        timeout=30
    )

print(f"\nPredict CSV response:")
print(f"Status Code: {response.status_code}")
print(f"Content-Type: {response.headers.get('content-type')}")
print(f"Content (first 1000 chars): {response.text[:1000]}")

Файл загружен. Строк: 966, колонок: 27
Успешно создано 966 результатов
INFO:     127.0.0.1:37150 - "POST /predict-csv?filename=heart_test.csv HTTP/1.1" 200 OK

Predict CSV response:
Status Code: 200
Content-Type: application/json
Content (first 1000 chars): {"status":"success","filename":"heart_test.csv","total_patients":966,"predictions":[{"patient_id":0,"prediction":0},{"patient_id":1,"prediction":1},{"patient_id":2,"prediction":1},{"patient_id":3,"prediction":0},{"patient_id":4,"prediction":0},{"patient_id":5,"prediction":0},{"patient_id":6,"prediction":0},{"patient_id":7,"prediction":0},{"patient_id":8,"prediction":0},{"patient_id":9,"prediction":1},{"patient_id":10,"prediction":1},{"patient_id":11,"prediction":1},{"patient_id":12,"prediction":1},{"patient_id":13,"prediction":1},{"patient_id":14,"prediction":0},{"patient_id":15,"prediction":0},{"patient_id":16,"prediction":1},{"patient_id":17,"prediction":1},{"patient_id":18,"prediction":1},{"patient_id":19,"prediction":1},{"patien